# Vertex AI Fine-Tuning with GroundCUA

Run this notebook in Google Colab or Vertex AI Workbench. Colab has ample disk (~50GB+) so the full GroundCUA download succeeds.

**Workflow:**
1. Install dependencies
2. Authenticate with GCP
3. Load GroundCUA from HuggingFace, convert to Vertex SFT format
4. Upload images + JSONL to GCS
5. Submit Vertex AI tuning job

## 1. Install dependencies

In [ ]:
!pip install -q datasets huggingface_hub google-cloud-aiplatform google-cloud-storage Pillow

## 2. Authenticate and config

In [ ]:
try:
  from google.colab import auth
  auth.authenticate_user()
  print("Colab: authenticated")
except ImportError:
  print("Not in Colab; ensure gcloud auth application-default login")

# --- CONFIG ---
PROJECT_ID = "your-gcp-project-id"
GCS_BUCKET = "your-gcs-bucket"
HF_TOKEN = None  # or "hf_xxx" if ServiceNow/GroundCUA is gated
LIMIT = 0  # 0 = all; set e.g. 1000 for quick test

## 3. Load GroundCUA and convert to Vertex format

In [ ]:
import io
import json
from collections import defaultdict
from google.cloud import storage

DATASET_ID = "ServiceNow/GroundCUA"

def bbox_center_to_normalized(bbox, width, height):
  if len(bbox) < 4:
    return 0.5, 0.5
  x1, y1, x2, y2 = bbox[0], bbox[1], bbox[2], bbox[3]
  cx = (x1 + x2) / 2.0
  cy = (y1 + y2) / 2.0
  x_norm = round(cx / width, 3) if width else 0.5
  y_norm = round(cy / height, 3) if height else 0.5
  return max(0.0, min(1.0, x_norm)), max(0.0, min(1.0, y_norm))

print("Loading dataset from HuggingFace...")
from datasets import load_dataset
ds = load_dataset(DATASET_ID, split="train", token=HF_TOKEN, trust_remote_code=True)

groups = defaultdict(list)
for row in ds:
  image_path = row.get("image_path")
  bbox = row.get("bbox")
  text = row.get("text") or row.get("label") or ""
  if not image_path or not bbox or len(bbox) < 4:
    continue
  groups[str(image_path)].append({"bbox": list(bbox), "text": str(text), "category": row.get("category")})

print(f"Grouped into {len(groups)} unique images")

# Upload images and build examples
from huggingface_hub import hf_hub_download
from PIL import Image

gcs_client = storage.Client(project=PROJECT_ID)
bucket = gcs_client.bucket(GCS_BUCKET)
gcs_prefix = f"gs://{GCS_BUCKET}/training/groundcua/images"

examples = []
count = 0
for image_path, entries in groups.items():
  if LIMIT and count >= LIMIT:
    break
  try:
    repo_path = f"images/{image_path}"
    local_path = hf_hub_download(repo_id=DATASET_ID, filename=repo_path, repo_type="dataset", token=HF_TOKEN)
    with open(local_path, "rb") as f:
      image_bytes = f.read()
    width, height = 1920, 1080
    try:
      img = Image.open(io.BytesIO(image_bytes))
      width, height = img.size
    except Exception:
      pass
    blob_name = f"training/groundcua/images/{image_path}"
    blob = bucket.blob(blob_name)
    blob.upload_from_string(image_bytes, content_type="image/png")
    url = f"{gcs_prefix}/{image_path}"
    for entry in entries:
      bbox = entry.get("bbox")
      text = entry.get("text") or ""
      if not bbox or len(bbox) < 4:
        continue
      x_norm, y_norm = bbox_center_to_normalized(bbox, width, height)
      instruction = f"Locate the element: {text}" if text else "Locate the element."
      examples.append({
        "messages": [
          {"role": "user", "content": [
            {"type": "image", "image_url": {"url": url}},
            {"type": "text", "text": instruction}
          ]},
          {"role": "model", "content": [{"type": "text", "text": f"point([{x_norm:.3f}, {y_norm:.3f}])"}]}
        ]
      })
      count += 1
      if LIMIT and count >= LIMIT:
        break
  except Exception as e:
    print(f"Skip {image_path}: {e}")

print(f"Built {len(examples)} training examples")

## 4. Upload JSONL to GCS

In [ ]:
import io

jsonl = "\n".join(json.dumps(ex) for ex in examples)
blob = bucket.blob("training/groundcua/dataset.jsonl")
blob.upload_from_string(jsonl, content_type="application/jsonl")
gcs_uri = f"gs://{GCS_BUCKET}/training/groundcua/dataset.jsonl"
print(f"Uploaded to {gcs_uri}")

## 5. Optional: merge with custom/traces

In [ ]:
# To merge with custom or traces JSONL, download those from GCS, concatenate examples, re-upload.
# Example:
# custom_uri = f"gs://{GCS_BUCKET}/training/custom/dataset.jsonl"
# combined = examples + [json.loads(ln) for ln in ...]
# Or run prepare_combined_dataset locally with --sources gs://.../groundcua/dataset.jsonl,gs://.../custom/dataset.jsonl
pass

## 6. Submit Vertex AI tuning job

In [ ]:
import vertexai
from vertexai.tuning import sft as vertex_sft

vertexai.init(project=PROJECT_ID, location="us-central1")
sft_job = vertex_sft.train(
  source_model="gemini-2.5-flash-001",
  train_dataset=gcs_uri,
  tuned_model_display_name="echoprism-global",
)
job_name = sft_job.resource_name
print(f"Job submitted: {job_name}")

## 7. Poll for completion

In [ ]:
job = vertex_sft.SupervisedTuningJob(job_name)
print(f"State: {job.state}")
# Re-run this cell to poll; when state is JOB_STATE_SUCCEEDED, get tuned_model_endpoint_name